# Make gpt-oss play games with Reinforcement Learning

This notebook demonstrates how you make `gpt-oss` play the 2048 game autonomously by using reinforcement learning (RL).

We will train `gpt-oss-20b` using [Unsloth](https://github.com/unslothai/unsloth) to develop a strategy for playing 2048. The strategy will run until the game ends, and the model will be rewarded or penalized based on whether it wins or loses.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/f9/2048_win.png/500px-2048_win.png" width=300 />

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Installation
To run `gpt-oss-20b` RL on a free Google Colab instance, we’ll use the GRPO algorithm along with [Unsloth](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning), an open-source tool that enables less VRAM usage and faster training.

In [2]:
import dataclasses
from dataclasses import dataclass

@dataclass
class CFG:
    project_name: str = "GRPO for 2048, 0.8. Unsloth 2026-01 again, gpt-oss-20b again"
    experiment_name: str = "Experiment #23, multiprocessing fix"
    lora_rank: int = 8
    reasoning_effort: str = "high"
    warmup_ratio: int = 0.02
    temperature: float = 1.0
    max_seq_length: int = 4096 - 512
    learning_rate: float = 5e-5
    beta: float = 0.0
    gradient_accumulation_steps: int = 16
    num_generations: int = 8
    importance_sampling_level: str = "sequence" # "token"
    loss_type: str = "grpo"
    patience_steps: int = 10
    # model_name: str = "unsloth/gpt-oss-20b-BF16"
    # model_name: str = "unsloth/gpt-oss-20b"
    model_name: str = "unsloth/gpt-oss-120b"
    # model_name: str = "unsloth/gpt-oss-20b-unsloth-bnb-4bit"
    # model_name: str = "unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit"
    # model_name: str = "unsloth/DeepSeek-R1-Distill-Qwen-14B-unsloth-bnb-4bit"
    # model_name: str = "unsloth/Qwen3-4B-Base"
    # model_name: str = "unsloth/gpt-oss-20b-BF16"
    fast_inference: bool = False
    number_of_runs_per_completion: int = 9
    per_device_train_batch_size: int = 1
    episode_limit_seconds: int = 600
    load_in_4bit: bool = True


# Чтобы сохранить конфигурацию текущего эксперимента, перенесём её в словарь cfg_dict
cfg = CFG()
cfg_dict = dataclasses.asdict(cfg)
cfg_dict

{'project_name': 'GRPO for 2048, 0.8. Unsloth 2026-01 again, gpt-oss-20b again',
 'experiment_name': 'Experiment #23, multiprocessing fix',
 'lora_rank': 8,
 'reasoning_effort': 'high',
 'warmup_ratio': 0.02,
 'temperature': 1.0,
 'max_seq_length': 3584,
 'learning_rate': 5e-05,
 'beta': 0.0,
 'gradient_accumulation_steps': 16,
 'num_generations': 8,
 'importance_sampling_level': 'sequence',
 'loss_type': 'grpo',
 'patience_steps': 10,
 'model_name': 'unsloth/gpt-oss-120b',
 'fast_inference': False,
 'number_of_runs_per_completion': 9,
 'per_device_train_batch_size': 1,
 'episode_limit_seconds': 600,
 'load_in_4bit': True}

We'll load gpt-oss-20b and set some parameters:
* `max_seq_length = 768` The maximum context length of the model. Increasing it will use more memory, and 768 was the maximum we found to fit on a free 15GB Tesla T4 machine
* `lora_rank = 4` The larger this number, the smarter the RL process, but the slower and more memory usage
* `load_in_4bit = True` Uses quantization to reduce memory usage by 75% without reducing accuracy that much. `load_in_16bit` will be faster but will need a 80GB GPU (H100, B200)
* `offload_embedding = True` Unsloth optimization which moves the embedding to CPU RAM, reducing VRAM by 1GB

In [3]:
import dataclasses
import mlflow

cfg_dict = dataclasses.asdict(cfg)

mlflow.set_experiment(cfg.project_name)
mlflow.start_run(run_name=cfg.experiment_name)

mlflow.log_params(cfg_dict)

To do efficient RL, we will use LoRA, which allows us to only add 1 to 5% of extra weights to the model for fine-tuning purposes. This allows us to save memory usage by 60% while retaining most accuracy. Read Unsloth's [gpt-oss RL Guide](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) for more details.

# 2048 game

We used GPT-5 to create a variant of the 2048 game. It should output the current game board state, and allow us to advance the game board state with 1 action (up, down, left, right).

In [4]:
from unsloth import FastLanguageModel
import torch

import os
# os.environ["UNSLOTH_DISABLE_SINKS"] = "1"
# os.environ["VLLM_DISABLE_SINKS"] = "1"
# os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = cfg.model_name,
    max_seq_length = cfg.max_seq_length,
    load_in_4bit=cfg.load_in_4bit,
    dtype = torch.bfloat16,
    fast_inference = cfg.fast_inference,
    offload_embedding = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/venv/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Gpt_Oss patching. Transformers: 4.57.3. vLLM: 0.18.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Workstation Edition. Num GPUs = 1. Max memory: 94.969 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu129. CUDA: 12.0. CUDA Toolkit: 12.9. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/16 [00:00<?, ?it/s]

Unsloth: Offloading embeddings to RAM to save 1.08 GB.


In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = cfg.lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = cfg.lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model` require gradients


In [ ]:
import torch, time
from torch.profiler import profile, ProfilerActivity

prompt_text = "Hello"
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

model.eval()
FastLanguageModel.for_inference(model)

with torch.inference_mode():
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=False) as prof:
        _ = model.generate(**inputs, max_new_tokens=64, use_cache=True)

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=30))

In [ ]:
#@title (Collapsible) 2048 Game Implementation
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
import random
import copy

def _compress_and_merge_row_left(row: List[int]) -> Tuple[List[int], int, bool]:
    n = len(row)
    tiles = [x for x in row if x != 0]
    gained = 0
    i = 0
    merged = []
    while i < len(tiles):
        if i + 1 < len(tiles) and tiles[i] == tiles[i + 1]:
            v = tiles[i] * 2
            gained += v
            merged.append(v)
            i += 2
        else:
            merged.append(tiles[i])
            i += 1
    merged += [0] * (n - len(merged))
    changed = merged != row
    return merged, gained, changed

def _move_left(board: List[List[int]]) -> Tuple[List[List[int]], int, bool]:
    changed_any = False
    total_gain = 0
    new_board = []
    for row in board:
        new_row, gained, changed = _compress_and_merge_row_left(row)
        new_board.append(new_row)
        total_gain += gained
        changed_any = changed_any or changed
    return new_board, total_gain, changed_any

def _move_right(board: List[List[int]]) -> Tuple[List[List[int]], int, bool]:
    changed_any = False
    total_gain = 0
    new_board = []
    for row in board:
        rev = list(reversed(row))
        new_rev, gained, changed = _compress_and_merge_row_left(rev)
        new_row = list(reversed(new_rev))
        new_board.append(new_row)
        total_gain += gained
        changed_any = changed_any or changed
    return new_board, total_gain, changed_any

def _transpose(board: List[List[int]]) -> List[List[int]]:
    return [list(row) for row in zip(*board)]

def _move_up(board: List[List[int]]) -> Tuple[List[List[int]], int, bool]:
    t = _transpose(board)
    moved, gain, changed = _move_left(t)
    return _transpose(moved), gain, changed

def _move_down(board: List[List[int]]) -> Tuple[List[List[int]], int, bool]:
    t = _transpose(board)
    moved, gain, changed = _move_right(t)
    return _transpose(moved), gain, changed

def _empty_cells(board: List[List[int]]) -> List[Tuple[int, int]]:
    size = len(board)
    return [(r, c) for r in range(size) for c in range(size) if board[r][c] == 0]

def _can_move(board: List[List[int]]) -> bool:
    if _empty_cells(board):
        return True
    size = len(board)
    for r in range(size):
        for c in range(size - 1):
            if board[r][c] == board[r][c + 1]:
                return True
    for r in range(size - 1):
        for c in range(size):
            if board[r][c] == board[r + 1][c]:
                return True
    return False

@dataclass
class GameBoard:
    size: int
    seed: Optional[int] = None
    target: int = 2048
    probability_fours: float = 0.10 # originally spawns (4) 10% of the time!
    _rng: random.Random = field(init=False, repr=False)
    _board: List[List[int]] = field(init=False, repr=False)
    _score: int = field(default=0, init=False, repr=False)
    _state: str = field(default="ongoing", init=False, repr=False)
    _won: bool = field(default=False, init=False, repr=False)

    def __post_init__(self):
        if self.size < 2:
            raise ValueError("Board size must be at least 2.")
        self._rng = random.Random(self.seed)
        self._board = [[0 for _ in range(self.size)] for _ in range(self.size)]
        self._add_random_tile()
        self._add_random_tile()
        self._update_state_after_change()

    class _BoardView:
        def __init__(self, game: "GameBoard"):
            self._game = game
        def __iter__(self):
            return iter(self._game._board)
        def __len__(self):
            return len(self._game._board)
        def __getitem__(self, idx):
            return self._game._board[idx]
        def __repr__(self) -> str:
            return repr(self._game._board)
        __str__ = __repr__
        def do_action(self, key: str) -> None:
            self._game.do_action(key)
            
        def state(self) -> str:
            return self._game.state()
            
        def pretty(self, colors: bool = True, border: bool = True, dot_for_zero: bool = True) -> str:
            return self._game._render_pretty(colors=colors, border=border, dot_for_zero=dot_for_zero)

    def board(self) -> "_BoardView":
        return GameBoard._BoardView(self)
        
    def state(self) -> str:
        return self._state

    def won(self) -> bool:  # optional helper for reward
        return self._won

    def score(self) -> int:
        return self._score
        
    def do_action(self, key: str) -> None:
        if self._state != "ongoing":
            return
    
        # normalize input
        if not isinstance(key, str) or not key:
            return
    
        k = key.strip().lower()
        move_map = {"a": _move_left, "d": _move_right, "w": _move_up, "s": _move_down}
    
        # invalid key => no-op (do nothing, keep state ongoing)
        if k not in move_map:
            return
    
        mover = move_map[k]
        new_board, gain, changed = mover(self._board)
        if changed:
            self._board = new_board
            self._score += gain
            self._add_random_tile()
        self._update_state_after_change()
        
    def _add_random_tile(self) -> bool:
        empties = _empty_cells(self._board)
        if not empties:
            return False
        r, c = self._rng.choice(empties)
        self._board[r][c] = 4 if self._rng.random() < self.probability_fours else 2
        return True
        
    def _update_state_after_change(self) -> None:
        # record win, but do NOT stop
        if (not self._won) and any(self.target in row for row in self._board):
            self._won = True

        # terminate only if no moves left
        if not _can_move(self._board):
            self._state = "terminated"
        else:
            self._state = "ongoing"
        
    def _render_pretty(self, colors: bool = True, border: bool = True, dot_for_zero: bool = True) -> str:
        """
        Pretty-print the board with colors that scale from 0 up to self.target.
        Uses ANSI 256-color codes (works in most terminals). Set colors=False to disable.
        """
        import math

        b = self._board
        mx = max((max(row) for row in b), default=0)
        cell_w = max(3, len(str(mx)))

        RESET = "\x1b[0m"

        # A smooth-ish gradient from cool → warm
        # (blue/cyan/green → yellow/orange/red). Tweak or expand as you like.
        GRAD = [33, 39, 45, 51, 50, 49, 48, 47, 46, 82, 118, 154, 190, 226, 220, 214, 208, 202, 196]
        ZERO_FG = 239  # dim gray

        def color_code(v: int) -> str:
            if not colors:
                return ""
            if v == 0:
                return f"\x1b[38;5;{ZERO_FG}m"
            # Normalize by exponent relative to target: r in [0,1]
            t = max(2, self.target)  # safety; avoid log2(1)
            # Guard: if v is not a power of two or is <1, handle gracefully
            try:
                r = max(0.0, min(1.0, math.log2(v) / math.log2(t)))
            except ValueError:
                r = 0.0
            idx = int(round(r * (len(GRAD) - 1)))
            return f"\x1b[38;5;{GRAD[idx]}m"

        def fmt(v: int) -> str:
            s = "." if (v == 0 and dot_for_zero) else str(v)
            s = s.rjust(cell_w)
            return color_code(v) + s + (RESET if colors else "")

        def hline(left: str, mid: str, right: str) -> str:
            return left + mid.join("─" * cell_w for _ in range(self.size)) + right

        rows = []
        if border:
            rows.append(hline("┌", "┬", "┐"))
        for r in range(self.size):
            content = "│".join(fmt(v) for v in b[r])
            rows.append(("│" + content + "│") if border else content)
            if border:
                rows.append(hline("└" if r == self.size - 1 else "├",
                                "┴" if r == self.size - 1 else "┼",
                                "┘" if r == self.size - 1 else "┤"))
        return "\n".join(rows)

For example let's create a board of size 5 X 5 and set the target to 8 instead of 2048.

**[NOTE]** 2048 originally spawns a (4) 10% of the time! We can disable this for harder games. See [Wikipedia page](https://en.wikipedia.org/wiki/2048_(video_game)) for more details.

In [ ]:
game = GameBoard(size = 5, seed = 42, target = 8, probability_fours = 0.10)
print(game.board().pretty(), game.state())

In [ ]:
game

We'll use WASD for the action space:

```
   W
A  S  D
```
Also `game.state()` will say `success` if we succeeded in getting the target!

In [ ]:
game.do_action("A")
print(game.board().pretty(), game.state())

In [ ]:
game.do_action("W")
print(game.board().pretty(), game.state())

In [ ]:
game.do_action("D")
print(game.board().pretty(), game.state())

In [ ]:
game.do_action("W")
print(game.board().pretty(), game.state())

In [ ]:
game.do_action("D")
print(game.board().pretty(), game.state())

If we do some other action that's not part of the action space, we will get an error, and the game will not accept anymore actions.

In [ ]:
game = GameBoard(size = 3, seed = 42, target = 8, probability_fours = 0.10)
game.do_action("AA") # Not in WASD
game.do_action("W")  # Doesn't do anything
game.do_action("A")  # Doesn't do anything
print(game.board().pretty(), game.state())

# RL Environment Setup

We'll set up a function to accept some strategy that'll emit an action within `WASD` and check the game state.

We'll also add a timer to only execute the stratgegy for 2 seconds maximum, otherwise it might never terminate!

In [ ]:
from dataclasses import dataclass
from typing import Callable
from unsloth import execute_with_time_limit

@dataclass
class Episode:
    game: "GameBoard"
    steps: int = 0

@execute_with_time_limit(cfg.episode_limit_seconds)
def run_episode(strategy: Callable, episode: Episode, patience_steps: int):
    stale_steps = 0

    while episode.game.state() == "ongoing":
        board_observation = [list(r) for r in episode.game.board()]
        action = strategy(board_observation)

        # detect state tampering (optional but recommended)
        if [list(r) for r in episode.game.board()] != board_observation:
            return episode.steps, "strategy_mutated_board", episode.game.won()

        board_before = board_observation
        score_before = episode.game.score()

        if isinstance(action, str) and action:
            episode.game.do_action(action)
        else:
            return episode.steps, "invalid_move", episode.game.won()

        board_after = tuple(tuple(r) for r in episode.game.board())
        score_after = episode.game.score()

        if board_after == board_before:
            return episode.steps, "invalid_move", episode.game.won()

        if score_after == score_before:
            stale_steps += 1
            if stale_steps >= patience_steps:
                return episode.steps, "stale_score_limit", episode.game.won()
        else:
            stale_steps = 0

        episode.steps += 1

    return episode.steps, episode.game.state(), episode.game.won()



def execute_strategy(strategy: Callable, game: "GameBoard", patience_steps: int):
    """
    Policy:
      - terminate if game.score() does not strictly improve for `patience_steps` steps
    """
    assert callable(strategy)
    if patience_steps <= 0:
        raise ValueError("patience_steps must be > 0")

    episode = Episode(game=game)

    try:
        return run_episode(strategy=strategy, episode=episode, patience_steps=patience_steps)
    except TimeoutError:
        return episode.steps, "step_timeout", episode.game.won()

Let's make a generic strategy to just hit `W`. We should expect this generic strategy to fail:

In [ ]:
def always_move_left(board):
    return "W"

game = GameBoard(size=8, seed=42, target=2048, probability_fours=0.10)

steps, state, won = execute_strategy(always_move_left, game, patience_steps=cfg.patience_steps)

if state == "step_limit":
    print(f"Stopped: hit step limit after {steps} steps")
elif state == "terminated":
    print(f"Terminated, won={won}, steps={steps}")
else:
    print(f"Finished: state={state}, won={won}, steps={steps}")

print(f"Score={game.score()}")
print(game.board().pretty())

# Code Execution

To execute and create a new Python function, we first have to check if the function does not call other global variables or cheat. This is called `countering reward hacking` since we don't want the function to cheat.

For example the below piece of code is fine, since it only imports Python level functions. We use `check_python_modules`:

In [ ]:
from unsloth import check_python_modules

sample = """
def strategy(board):
    import math
    from typing import Callable
    return "W"
"""
ok, info = check_python_modules(sample)
print("Only Python imports?", ok)
print(info)

For the below piece of code, since we import `numpy`, we should not allow the execution:

In [ ]:
sample = """
def strategy(board):
    from numpy import matmul
    return "W"
"""
ok, info = check_python_modules(sample)
print("Only Python imports?", ok)
print(info)

We also disallow global variable access. We'll use Unsloth's `create_locked_down_function` function


In [ ]:
from unsloth import create_locked_down_function
function = """
def import_numpy():
    np.matmul
    print("Success")
"""
f = create_locked_down_function(function)
try:
    f()
except Exception as e:
    print(str(e))

In [ ]:
from unsloth import create_locked_down_function
function = """
def add(a, b):
    def adder(a):
        return a + b
    return adder(b) + b
"""
f = create_locked_down_function(function)
try:
    print(f(10, 20))
except Exception as e:
    print(str(e))

# Data & RL task setup

We now have to create a prompt to tell the model to create a strategy for the 2048 game. You can customize this to some other task for another RL task.

First, let's prompt gpt-oss without RL and see how it goes:

In [ ]:
prompt = """
Act as a senior software developer, expert in performance optimization.
Create 2048 strategy using only native Python code.
You are given a list of list of numbers for the current board state.
Output one action for "W", "A", "S", "D" on what is the optimal next step.
Chance of spawning 2 is 90% and 4 is 10% and it is critical to model this.
Board size is strictly 4x4.
Output your function in backticks using the format below:
```python
def strategy(board):
    return "W" # Example
```
Any helper functions (if needed) should be inside def strategy, meaning strategy is the only top-level function allowed and all neccesary code including imports must be inside strategy.
Use techniques like bithacking to speedup execution.
""".strip()

# prompt = """
# You are writing a policy function for a 2048-like game and your task is to make best next move based on current state of board.

# SIGNATURE:
# def strategy(board):
#     return <NEXT BEST MOVE>

# INPUT:
#   - board is a rectangular list of lists of integers (size may be 4x4, 6x6, etc)

# RETURN VALUE:
#   - return exactly one character: 'W', 'A', 'S', or 'D', meaning up, left, down, right accordingly.

# OUTPUT FORMAT — STRICT:
#   - return ONLY valid Python code. No markdown, no backticks.
#   - the first line MUST be exactly: def strategy(board):
#   - only ONE top-level function is allowed: strategy. All helpers must be nested inside strategy.
#   - No imports. No global variables. No docstrings.

  
# IMPORTANT:
# - Board is a copy, you can change it, but your next move is the only thing that matters to Game interpeter.
# - Game interpreter will call strategy multiple times limiting all game with 2 seconds, so don't overthink
# - Keep solution short, like 150 lines of code max.


# """.strip()

print(prompt)

# Reward functions

We now design a `extract_function` function which simply extracts the function wrapped in 3 back ticks.

And 3 reward functions:

1. `function_works` which rewards the model if the strategy is a valid Python function.
2. `no_cheating` which checks if the function imported other modules, and if it did, we penalize it.
3. `strategy_succeeds` which checks if the game strategy actually succeeds in attaining 2048 after running the auto-generated strategy.

In [ ]:
import re

def extract_function(text):
    if text.count("```") >= 2:
        second = text.rfind("```")
        first = text.rfind("```", 0, second)
        if first != -1:
            fx = text[first + 3:second].strip()
            fx = fx.removeprefix("python\n").strip()
            def_pos = fx.find("def")
            if def_pos != -1:
                fx = fx[def_pos:]
                if fx.startswith("def strategy(board):"):
                    return fx
    return None

print(extract_function(prompt))

# def extract_function(text):
#     if text.count("```") >= 2:
#         first = text.find("```") + 3
#         second = text.find("```", first)
#         fx = text[first:second].strip()
#         fx = fx.removeprefix("python\n")
#         fx = fx[fx.find("def"):]
#         if fx.startswith("def strategy(board):"):
#             return fx
#     return None

# print(extract_function(prompt))

Below is our `function_works` reward function which uses Python's `exec` but guarded by not allowing leakage of local and global variables. We can also use `check_python_modules` first to check if there are errors before even executing the function:

In [ ]:
ok, info = check_python_modules("def a")
ok, info

In [ ]:
def function_works(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        function = extract_function(response)

        if function is None:
            scores.append(-5.0)
            continue

        ok, info = check_python_modules(function)
        if (not ok) or ("error" in str(info).lower()):
            scores.append(-10.0)
            continue

        try:
            _ = create_locked_down_function(function)
            scores.append(0.0)   # <- KEY: no “free” positive reward
        except:
            scores.append(-5.0)
    return scores

`no_cheating` checks if the function cheated since it might have imported Numpy or other functions:

In [ ]:
def no_cheating(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        function = extract_function(response)

        if function is None:
            scores.append(0.0)   # function_works already penalizes missing
            continue

        ok, _ = check_python_modules(function)
        scores.append(0.0 if ok else -50.0)  # only big negative on violation
    return scores

In [ ]:
# ============================================================
# Imports
# ============================================================

import os
import time
import traceback
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import dataclass
from typing import List, Optional, Literal, Dict, Tuple

import numpy as np
import mlflow


# ============================================================
# Assumed to already exist in your notebook:
#   - cfg.patience_steps
#   - cfg.number_of_runs_per_completion
#   - extract_function
#   - check_python_modules
#   - create_locked_down_function
#   - GameBoard
#   - execute_strategy(strategy, game, patience_steps)
#
# Also assumed:
#   - An MLflow run is already active (mlflow.start_run(...) called earlier)
# ============================================================


# ============================================================
# Episode counter (global, monotonic)
# ============================================================

class EpisodeCounter:
    def __init__(self, start: int = 0):
        self.value = int(start)

    def next(self) -> int:
        v = self.value
        self.value += 1
        return v


EPISODE_COUNTER = EpisodeCounter(start=0)


# ============================================================
# Data models
# ============================================================

Status = Literal["ok", "invalid", "exception"]


@dataclass(frozen=True)
class RunResult:
    seed: int
    steps: int
    won: bool
    game_score: float
    run_duration: float
    board_pretty: str


@dataclass(frozen=True)
class AggregateResult:
    max_run_duration: float
    win_rate: float
    best_game_score: float
    worst_game_score: float
    median_game_score: float
    best_board_pretty: Optional[str]


@dataclass(frozen=True)
class EvalResult:
    status: Status
    duration: float                 # total wall time (parent-side measurement)
    aggregate: Optional[AggregateResult]
    error: Optional[str] = None


# ============================================================
# Concern A: Strategy building / validation
# ============================================================

def _build_strategy_from_response(*, response: str):
    function = extract_function(response)
    if function is None:
        return "invalid", None

    ok, info = check_python_modules(function)
    if not ok or "error" in str(info).lower():
        return "invalid", None

    return "ok", create_locked_down_function(function)


# ============================================================
# Concern B: Single run evaluation
# ============================================================

def _run_one_game(*, strategy, seed: int, patience_steps: int) -> RunResult:
    run_started = time.perf_counter()

    game = GameBoard(
        size=4,
        seed=int(seed),
        target=2048,
        probability_fours=0.10,
    )

    steps, _termination_reason, won = execute_strategy(
        strategy,
        game,
        patience_steps=int(patience_steps),
    )

    run_duration = time.perf_counter() - run_started
    game_score = float(game.score() if hasattr(game, "score") else 0.0)
    board_pretty = game.board().pretty()

    return RunResult(
        seed=int(seed),
        steps=int(steps),
        won=bool(won),
        game_score=float(game_score),
        run_duration=float(run_duration),
        board_pretty=str(board_pretty),
    )


# ============================================================
# Concern C: Aggregation
# ============================================================

def _aggregate_runs(*, runs: List[RunResult]) -> AggregateResult:
    if not runs:
        return AggregateResult(
            max_run_duration=0.0,
            win_rate=0.0,
            best_game_score=0.0,
            worst_game_score=0.0,
            median_game_score=0.0,
            best_board_pretty=None,
        )

    max_run_duration = max(r.run_duration for r in runs)
    win_rate = sum(1 for r in runs if r.won) / len(runs)

    best_run = max(runs, key=lambda r: r.game_score)
    worst_run = min(runs, key=lambda r: r.game_score)

    scores = np.array([r.game_score for r in runs], dtype=float)
    median_game_score = float(np.median(scores)) if len(scores) else 0.0

    return AggregateResult(
        max_run_duration=float(max_run_duration),
        win_rate=float(win_rate),
        best_game_score=float(best_run.game_score),
        worst_game_score=float(worst_run.game_score),
        median_game_score=float(median_game_score),
        best_board_pretty=best_run.board_pretty,
    )


# ============================================================
# Worker entry: "one run" task (fine-grained for dynamic scheduling)
# ============================================================

def _eval_one_run(
    *,
    response: str,
    seed: int,
    patience_steps: int,
) -> Tuple[Status, Optional[RunResult], Optional[str]]:
    try:
        status, strategy = _build_strategy_from_response(response=response)
        if status != "ok":
            return "invalid", None, None

        rr = _run_one_game(
            strategy=strategy,
            seed=int(seed),
            patience_steps=int(patience_steps),
        )
        return "ok", rr, None

    except Exception as exc:
        return "exception", None, f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}"


# ============================================================
# Parent-side helpers
# ============================================================

def _extract_responses(*, completions) -> List[str]:
    responses: List[str] = []
    for completion in completions:
        try:
            response = completion[0].get("content", "") if completion and completion[0] else ""
        except Exception:
            response = ""
        responses.append(response)
    return responses


def _make_seeds(*, n: int, number_of_runs_per_completion: int) -> np.ndarray:
    return np.random.randint(
        0,
        10_000,
        size=(n, number_of_runs_per_completion),
        dtype=np.int64,
    )


def _run_parallel_evals_flat(
    *,
    responses: List[str],
    seeds_matrix: np.ndarray,  # shape: (n_responses, runs_per_response)
    patience_steps: int,
) -> List[EvalResult]:
    """
    Dynamic scheduling:
      - one task per (response, seed)
      - fixed-size process pool (<= CPU count)
      - as soon as a worker is free, it takes the next task
    """
    n = len(responses)
    runs_by_resp: List[List[RunResult]] = [[] for _ in range(n)]
    worst_status_by_resp: List[Status] = ["ok" for _ in range(n)]
    error_by_resp: List[Optional[str]] = [None for _ in range(n)]

    total_tasks = int(seeds_matrix.size)
    max_workers = min(os.cpu_count() or 1, total_tasks)

    parent_started = time.perf_counter()

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_resp_idx: Dict = {}

        for i in range(n):
            for seed in seeds_matrix[i].tolist():
                fut = executor.submit(
                    _eval_one_run,
                    response=responses[i],
                    seed=int(seed),
                    patience_steps=int(patience_steps),
                )
                future_to_resp_idx[fut] = i

        for fut in as_completed(future_to_resp_idx):
            i = future_to_resp_idx[fut]
            status, rr, err = fut.result()

            if status == "exception":
                worst_status_by_resp[i] = "exception"
                if error_by_resp[i] is None:
                    error_by_resp[i] = err
            elif status == "invalid" and worst_status_by_resp[i] == "ok":
                worst_status_by_resp[i] = "invalid"

            if rr is not None:
                runs_by_resp[i].append(rr)

    parent_duration = time.perf_counter() - parent_started

    eval_results: List[EvalResult] = []
    for i in range(n):
        status = worst_status_by_resp[i]
        if status != "ok":
            eval_results.append(
                EvalResult(
                    status=status,
                    duration=float(parent_duration),
                    aggregate=None,
                    error=error_by_resp[i],
                )
            )
        else:
            aggregate = _aggregate_runs(runs=runs_by_resp[i])
            eval_results.append(
                EvalResult(
                    status="ok",
                    duration=float(parent_duration),
                    aggregate=aggregate,
                )
            )

    return eval_results


def _format_episode_block(*, episode_idx: int, response: str, eval_result: EvalResult) -> str:
    lines: List[str] = []
    lines.append("\n" + "=" * 110)
    lines.append(f"EPISODE {episode_idx}")
    lines.append("-" * 110)
    lines.append(((response or "").strip() or "<EMPTY>"))

    if eval_result.status == "invalid":
        if extract_function(response) is None:
            lines.append("<NO FUNCTION EXTRACTED>")
        lines.append("RESULT: INVALID (module check failed)")
        return "\n".join(lines)

    if eval_result.status == "exception":
        lines.append(f"RESULT: EXCEPTION after {eval_result.duration:.4f}s")
        lines.append(eval_result.error or "")
        return "\n".join(lines)

    agg = eval_result.aggregate
    assert agg is not None

    lines.append(
        f"RESULT: win_rate={100.0 * agg.win_rate:.1f}%  "
        f"best_game_score={agg.best_game_score:.1f}  "
        f"worst_game_score={agg.worst_game_score:.1f}  "
        f"median_game_score={agg.median_game_score:.1f}  "
        f"max_run_duration={agg.max_run_duration:.4f}s  "
        f"worker_time={eval_result.duration:.4f}s"
    )

    if agg.best_board_pretty:
        lines.append("")
        lines.append(f"BEST BOARD (score={agg.best_game_score:.1f}):")
        lines.append(agg.best_board_pretty)

    return "\n".join(lines)


def _report_mlflow(*, episode_idx: int, eval_result: EvalResult) -> None:
    # Ensure we have an active run; otherwise logging will be a no-op / error depending on setup
    mlflow.log_metric("episode/duration", eval_result.duration, step=episode_idx)

    if eval_result.status != "ok" or eval_result.aggregate is None:
        mlflow.set_tag(f"episode/{episode_idx}/status", eval_result.status)
        if eval_result.error:
            mlflow.log_text(eval_result.error, f"episode_{episode_idx}_error.txt")
        return

    agg = eval_result.aggregate
    mlflow.log_metric("episode/max_run_duration", agg.max_run_duration, step=episode_idx)
    mlflow.log_metric("episode/win_rate", agg.win_rate, step=episode_idx)
    mlflow.log_metric("episode/best_game_score", agg.best_game_score, step=episode_idx)
    mlflow.log_metric("episode/worst_game_score", agg.worst_game_score, step=episode_idx)
    mlflow.log_metric("episode/median_game_score", agg.median_game_score, step=episode_idx)

    if agg.best_board_pretty:
        mlflow.log_text(agg.best_board_pretty, f"episode_{episode_idx}_best_board.txt")


def _returned_score_median(eval_result: EvalResult) -> float:
    if eval_result.status == "ok" and eval_result.aggregate is not None:
        return float(eval_result.aggregate.median_game_score)
    if eval_result.status == "exception":
        return -3.0
    return 0.0


# ============================================================
# Main entry point
# ============================================================

def strategy_succeeds(completions, **kwargs):
    number_of_runs_per_completion = int(cfg.number_of_runs_per_completion)
    patience_steps = int(cfg.patience_steps)

    responses = _extract_responses(completions=completions)

    seeds_matrix = _make_seeds(
        n=len(responses),
        number_of_runs_per_completion=number_of_runs_per_completion,
    )

    eval_results = _run_parallel_evals_flat(
        responses=responses,
        seeds_matrix=seeds_matrix,
        patience_steps=patience_steps,
    )

    scores: List[float] = [0.0] * len(responses)

    for i, (response, eval_result) in enumerate(zip(responses, eval_results)):
        episode_idx = EPISODE_COUNTER.next()

        episode_text = _format_episode_block(
            episode_idx=episode_idx,
            response=response,
            eval_result=eval_result,
        )
        

        mlflow.log_text(episode_text, f"episode_{episode_idx:06d}_code.txt")

        print(episode_text, f"episode_{episode_idx:06d}_code.txt")

        _report_mlflow(episode_idx=episode_idx, eval_result=eval_result)

        scores[i] = _returned_score_median(eval_result)

    return scores

Next `strategy_succeeds` checks if the strategy actually allows the game to terminate. Imagine if the strategy simply returned "W" which would fail after a time limit of 10 seconds.

We also add a global `PRINTER` to print out the strategy and board state.

We'll now create the dataset which includes a replica of our prompt. Remember to add a reasoning effort of low! You can choose high reasoning mode, but this'll only work on more memory GPUs like H100s.

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations! We also support GSPO, GAPO, Dr GRPO and more! Go the Unsloth [Reinforcement Learning Docs](https://docs.unsloth.ai/get-started/reinforcement-learning-rl-guide) for more options.

In [ ]:
from datasets import Dataset
dataset = Dataset.from_list([{"prompt" : [{"role": "user", "content": prompt.strip()}], "answer" : 0, "reasoning_effort": cfg.reasoning_effort, }]*1000)
maximum_length = len(tokenizer.apply_chat_template([{"role": "user", "content": prompt.strip()}], add_generation_prompt = True))
print(maximum_length)
dataset[0]

In [ ]:
from transformers import GenerationConfig
from trl import GRPOConfig, GRPOTrainer

max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = cfg.max_seq_length - max_prompt_length

generation_config = GenerationConfig(
    max_new_tokens = max_completion_length,
    temperature    = cfg.temperature,
    eos_token_id   = None,
)

training_args = GRPOConfig(
    learning_rate = cfg.learning_rate,
    beta=cfg.beta,
    weight_decay = 0.01,
    warmup_ratio = cfg.warmup_ratio,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    # optim = "paged_adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = cfg.per_device_train_batch_size,
    gradient_accumulation_steps = cfg.gradient_accumulation_steps, # Increase to 4 for smoother training
    num_generations = cfg.num_generations, # Decrease if out of memory
    # max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 1000,
    save_steps = 100,
    report_to = "none", # Can use Weights & Biases, TrackIO
    output_dir = "outputs",
    mask_truncated_completions = True,
    importance_sampling_level = cfg.importance_sampling_level,
    loss_type = cfg.loss_type,

    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

In [ ]:
import time
import mlflow
from transformers import TextStreamer

# Build prompt
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
    reasoning_effort=cfg.reasoning_effort,
)

# Tokenize
inputs = tokenizer(text, return_tensors="pt").to("cuda")
prompt_tokens = inputs["input_ids"].shape[-1]

# Time generation (no CUDA sync)
start_time = time.time()

outputs = model.generate(
    **inputs,
    generation_config=generation_config,
    streamer=TextStreamer(tokenizer, skip_prompt=False),
    return_dict_in_generate=True,
    output_scores=False,
)

end_time = time.time()

# Tokens/sec
total_tokens = outputs.sequences.shape[-1]
new_tokens = total_tokens - prompt_tokens
elapsed = end_time - start_time

tokens_per_sec = new_tokens / elapsed if elapsed > 0 else 0.0

# Report to MLflow (to the active run)
mlflow.log_metric("generation/tokens_per_sec", tokens_per_sec, step=0)  # replace step if needed

print(f"{tokens_per_sec:.2f} tokens/sec")

And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        function_works,
        no_cheating,
        strategy_succeeds,
    ],
    args = training_args,
    train_dataset = dataset,
    generation_config = generation_config,
    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)

And let's train the model!

**NOTE** A T4 free GPU might take 5 minutes for one generation sadly since it's an old GPU - A100 or H100 will be much faster!

In [ ]:
trainer.train()

<a name="Inference"></a>
# Inference
Now let's try the model we just trained!

In [ ]:
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize = False,
    add_generation_prompt = True,
    reasoning_effort = cfg.reasoning_effort,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 1.0,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

<a name="Save"></a>
### Saving to float16 or `MXFP4`

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `mxfp4` for MXFP4 (OpenAI's gpt-oss native precision). We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge and push to hub in mxfp4 4bit format
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method = "mxfp4")
if False:
    model.push_to_hub_merged("repo_id/repo_name", tokenizer, token = "hf...", save_method = "mxfp4")

# Merge and push to hub in 16bit
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method = "merged_16bit")
if False: # Pushing to HF Hub
    model.push_to_hub_merged("hf/gpt-oss-finetune", tokenizer, save_method = "merged_16bit", token = "")

# And we're done!
Congratulations you just learned how to do reinforcement learning with gpt-oss! There were some advanced topics explained in this notebook - to learn more about gpt-oss and RL, there are more docs in Unsloth's [Reinforcement Learning Guide with gpt-oss](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning)

In [ ]:
mlflow.end_run()